# LLM Generation via Groq and LangChain

This notebook focuses on the generation component of the RAG pipeline, demonstrating how to utilize Groq-hosted Large Language Models (LLMs) to synthesize answers from **actual retrieved context**.

## 1. Groq LLM Configuration

We utilize the `langchain-groq` integration to interface with high-performance inference endpoints. The system is configured for low-latency response generation.

In [ ]:
import sys
sys.path.append('..')

from src.pipeline import build_groq_llm

# Initialize the LLM client
llm = build_groq_llm()
print(f"Groq Client Initialized. Model: {llm.model_name}")

Groq Client Initialized. Model: llama-3.1-8b-instant


## 2. Integrated RAG Synthesis

The `AdvancedRAGPipeline` orchestrates the final generation step. Unlike basic LLM calls, this pipeline first retrieves context from a live vector store (Chroma in this example) and then constructs an optimized prompt to synthesize a grounded response.

In [ ]:
from src.embeddings import EmbeddingManager
from src.vectorstores import ChromaVectorStore
from src.retrieval import ChromaRAGRetriever
from src.pipeline import AdvancedRAGPipeline

# 1. Initialize Retrieval Components
embed_manager = EmbeddingManager()
embed_manager.load_model()

vector_store = ChromaVectorStore()
retriever = ChromaRAGRetriever(vector_store, embed_manager)

# 2. Initialize orchestration pipeline
pipeline = AdvancedRAGPipeline(retriever=retriever, llm=llm)

# 3. Execute an end-to-end query
query = "What skills is the interviewer looking for?"
result = pipeline.query(query, top_k=3, min_score=0.1, summarize=True)

print(f"--- GENERATION RESULTS ---\n")
print(f"QUERY: {result['question']}")
print(f"\nFINAL ANSWER:\n{result['answer']}")

if result['summary']:
    print(f"\nSUMMARY: {result['summary']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3600.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- GENERATION RESULTS ---

QUERY: What skills is the interviewer looking for?

FINAL ANSWER:
The interviewer is looking for three main skills:

1. Analytic Skills
2. Coding Skills
3. Technical knowledge

Citations:
[1] cheatsheet.pdf (page 0)
[2] cheatsheet.pdf (page 0)
[3] cheatsheet.pdf (page 0)

SUMMARY: However, there is no answer provided for me to summarize. If you could provide the answer, I would be happy to assist you in summarizing it in 2 sentences.


## Summary

This demonstration shows the complete RAG lifecycle: the `AdvancedRAGPipeline` successfully retrieved documentation from the vector store and used the Groq LLM to generate a professional, cited answer. This approach ensures that the model's output is grounded in your specific dataset rather than generic training data.